In [1]:
# ============================================================
# EDA Starter Kit for VAR Spillover Data
# File: ./only_var_input.csv
# Output: ./histogram
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.stattools import adfuller, acf, pacf

# ----------------------------
# 0) Config
# ----------------------------
DATA_PATH = "./only_var_input.csv"
OUT_DIR = "./histogram"
os.makedirs(OUT_DIR, exist_ok=True)

ROLLING_WINDOWS = [20, 60]
ACF_LAGS = 40
PACF_LAGS = 40
ADF_AUTOLAG = "AIC"

# ----------------------------
# 1) Load
# ----------------------------
df = pd.read_csv(DATA_PATH)
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

num_cols = [c for c in df.columns if c != "Date"]

print("Shape:", df.shape)
print("Date range:", df["Date"].min(), "~", df["Date"].max())
print("Columns:", df.columns.tolist())

# ----------------------------
# 2) Basic checks
# ----------------------------
miss = df[num_cols].isna().sum().sort_values(ascending=False)
print("\n=== Missing counts ===")
print(miss)

desc = df[num_cols].describe().T
desc["skew"] = df[num_cols].skew()
desc["kurtosis"] = df[num_cols].kurtosis()
desc.to_csv(os.path.join(OUT_DIR, "desc_stats.csv"))

z = (df[num_cols] - df[num_cols].mean()) / df[num_cols].std(ddof=0)
outlier_any = (z.abs() > 5).any(axis=1)
df.loc[outlier_any, ["Date"] + num_cols].to_csv(
    os.path.join(OUT_DIR, "outlier_rows_z_gt_5.csv"), index=False
)

# ----------------------------
# 3) Correlation matrix + heatmap (with values)
# ----------------------------
corr = df[num_cols].corr()
corr.to_csv(os.path.join(OUT_DIR, "corr.csv"))

plt.figure(figsize=(8, 6))
im = plt.imshow(corr.values, aspect="auto", cmap="coolwarm", vmin=-1, vmax=1)

plt.xticks(range(len(num_cols)), num_cols, rotation=45, ha="right")
plt.yticks(range(len(num_cols)), num_cols)
plt.title("Correlation Heatmap")

# 숫자 표시
for i in range(len(num_cols)):
    for j in range(len(num_cols)):
        plt.text(j, i, f"{corr.values[i, j]:.2f}",
                 ha="center", va="center", fontsize=8, color="black")

plt.colorbar(im)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "corr_heatmap.png"), dpi=200)
plt.close()

# ----------------------------
# 4) Time series plots
# ----------------------------
for col in num_cols:
    plt.figure(figsize=(12, 4))
    plt.plot(df["Date"], df[col])
    plt.title(f"Time Series - {col}")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f"ts__{col}.png"), dpi=200)
    plt.close()

# ----------------------------
# 5) Histogram plots
# ----------------------------
for col in num_cols:
    plt.figure(figsize=(8, 4))
    plt.hist(df[col].dropna(), bins=60)
    plt.title(f"Distribution - {col}")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f"hist__{col}.png"), dpi=200)
    plt.close()

# ----------------------------
# 6) Rolling volatility
# ----------------------------
for w in ROLLING_WINDOWS:
    for col in num_cols:
        rv = df[col].rolling(w).std()
        plt.figure(figsize=(12, 4))
        plt.plot(df["Date"], rv)
        plt.title(f"Rolling Std ({w}) - {col}")
        plt.tight_layout()
        plt.savefig(os.path.join(OUT_DIR, f"rollstd{w}__{col}.png"), dpi=200)
        plt.close()

# ----------------------------
# 7) ACF / PACF
# ----------------------------
def stem_plot(vals, title, out_path):
    x = np.arange(len(vals))
    plt.figure(figsize=(10, 4))
    plt.stem(x, vals)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()

for col in num_cols:
    x = df[col].dropna().values
    acf_vals = acf(x, nlags=ACF_LAGS, fft=True)
    pacf_vals = pacf(x, nlags=PACF_LAGS, method="ywm")

    stem_plot(acf_vals, f"ACF - {col}",
              os.path.join(OUT_DIR, f"acf__{col}.png"))
    stem_plot(pacf_vals, f"PACF - {col}",
              os.path.join(OUT_DIR, f"pacf__{col}.png"))

# ----------------------------
# 8) ADF test
# ----------------------------
adf_rows = []
for col in num_cols:
    x = df[col].dropna().values
    res = adfuller(x, autolag=ADF_AUTOLAG)
    adf_rows.append({
        "col": col,
        "adf_stat": res[0],
        "pvalue": res[1],
        "used_lag": res[2],
        "nobs": res[3]
    })

adf_df = pd.DataFrame(adf_rows).sort_values("pvalue")
adf_df.to_csv(os.path.join(OUT_DIR, "adf_results.csv"), index=False)

print("\nEDA finished. Outputs saved in:", OUT_DIR)

Shape: (1325, 6)
Date range: 2020-10-13 00:00:00 ~ 2026-01-12 00:00:00
Columns: ['Date', 'dlog_SOLVPN', 'dlog_COPPER', 'dlog_DXY', 'd_UST10Y', 'dlog_VIX']

=== Missing counts ===
dlog_SOLVPN    0
dlog_COPPER    0
dlog_DXY       0
d_UST10Y       0
dlog_VIX       0
dtype: int64

EDA finished. Outputs saved in: ./histogram
